In [2]:
import requests
import pandas as pd
import os
from geopy.geocoders import Nominatim
import time

In [48]:

overpass_url = "http://overpass-api.de/api/interpreter"

print("=== Getting all suburbs in Victoria, Australia ===")

# Query for all suburb-level administrative boundaries in Victoria
# We'll use a bounding box that covers the Greater Melbourne area
# Approximate bounding box for Greater Melbourne: 
# South: -38.5, West: 144.5, North: -37.3, East: 145.8
melbourne_bbox = "-38.2255,144.18,-37.5,145.375"
#"-38.2255,144.18,-37.5,145.375"

query_suburbs = f"""
[out:json][timeout:120];
(
  relation["admin_level"="10"]["boundary"="administrative"]({melbourne_bbox});
  relation["admin_level"="9"]["place"="suburb"]({melbourne_bbox});
  relation["place"="suburb"]({melbourne_bbox});
  // Ways tagged as suburbs
  way["place"="suburb"]({melbourne_bbox});    

  // Nodes tagged as suburbs  
  node["place"="suburb"]({melbourne_bbox});
);
out ids tags;
"""

print("Querying for all suburbs in the Melbourne metropolitan area...")
try:
    response = requests.get(overpass_url, params={'data': query_suburbs})
    response.raise_for_status()
    suburbs_data = response.json()
except requests.exceptions.RequestException as e:
    print(f"An error occurred while fetching suburbs: {e}")
    suburbs_data = {'elements': []}

print(f"Found {len(suburbs_data['elements'])} suburbs in the Melbourne area.")

if suburbs_data['elements']:
    all_supermarkets = {}
    
    # Process each suburb
    for i, suburb in enumerate(suburbs_data['elements']):
        suburb_id = suburb['id']
        suburb_type = suburb['type']
        suburb_name = suburb['tags'].get('name', f'Unnamed Suburb (ID: {suburb_id})')
        
        # Calculate area ID (all our results are relations based on the query)
        area_id = 3600000000 + suburb_id
        
        print(f"\n[{i+1}/{len(suburbs_data['elements'])}] Fetching supermarkets in {suburb_name}...")
        
        query_supermarkets = f"""
        [out:json][timeout:30];
        area({area_id});
        (
          node["shop"="supermarket"](area);
          way["shop"="supermarket"](area);
          relation["shop"="supermarket"](area);
        );
        out center;
        """
        
        try:
            supermarket_response = requests.get(overpass_url, params={'data': query_supermarkets})
            supermarket_response.raise_for_status()
            supermarket_data = supermarket_response.json()
            
            if supermarket_data['elements']:
                all_supermarkets[suburb_name] = []
                for element in supermarket_data['elements']:
                    name = element['tags'].get('name', 'Unnamed Supermarket')
                    if 'center' in element:
                        lat, lon = element['center']['lat'], element['center']['lon']
                    else:
                        lat, lon = element['lat'], element['lon']
                    
                    supermarket_details = {'name': name, 'lat': lat, 'lon': lon}
                    all_supermarkets[suburb_name].append(supermarket_details)
                    print(f"  ✓ {name}")
                print(f"  → Found {len(supermarket_data['elements'])} supermarkets")
            else:
                print(f"  → No supermarkets found")

        except requests.exceptions.RequestException as e:
            print(f"  ✗ Error fetching supermarkets for {suburb_name}: {e}")

        # Be polite to the API
        time.sleep(1)

print(f"\n\n=== SUMMARY ===")
print(f"Processed {len(suburbs_data['elements'])} suburbs")
print(f"Found supermarkets in {len(all_supermarkets)} suburbs")

total_supermarkets = sum(len(markets) for markets in all_supermarkets.values())
print(f"Total supermarkets found: {total_supermarkets}")

# Show some examples
print(f"\nSuburbs with supermarkets:")
for suburb, markets in list(all_supermarkets.items())[:10]:  # Show first 10
    print(f"  {suburb}: {len(markets)} supermarkets")

=== Getting all suburbs in Victoria, Australia ===
Querying for all suburbs in the Melbourne metropolitan area...
Found 594 suburbs in the Melbourne area.

[1/594] Fetching supermarkets in Heathmont...
  → No supermarkets found

[2/594] Fetching supermarkets in Dandenong...
  → No supermarkets found

[3/594] Fetching supermarkets in Lilydale...
  → No supermarkets found

[4/594] Fetching supermarkets in Wheelers Hill...
  → No supermarkets found

[5/594] Fetching supermarkets in Grovedale...
  → No supermarkets found

[6/594] Fetching supermarkets in Manifold Heights...
  → No supermarkets found

[7/594] Fetching supermarkets in Fyansford...
  → No supermarkets found

[8/594] Fetching supermarkets in Oakleigh South...
  → No supermarkets found

[9/594] Fetching supermarkets in Box Hill North...
  → No supermarkets found

[10/594] Fetching supermarkets in Carlton North...
  → No supermarkets found

[11/594] Fetching supermarkets in Strathmore...
  → No supermarkets found

[12/594] Fetch

KeyboardInterrupt: 

In [14]:
all_supermarkets['Caulfield North']

[{'name': 'Woolworths Metro', 'lat': -37.8630952, 'lon': 145.0101814},
 {'name': 'Coles', 'lat': -37.8760273, 'lon': 145.0376175}]

In [ ]:
import pandas as pd
from datetime import datetime

print(f"Starting comprehensive amenity collection at {datetime.now()}")
print("Collecting supermarkets, cafes, gyms, and libraries for Greater Melbourne...")
print("This will take approximately 15-20 minutes.")

# Define all amenity types and their queries
amenity_queries = {
    'supermarket': '''
        (
          node["shop"="supermarket"](area);
          way["shop"="supermarket"](area);
          relation["shop"="supermarket"](area);
        );
    ''',
    'cafe': '''
        (
          node["amenity"="cafe"](area);
          way["amenity"="cafe"](area);
          relation["amenity"="cafe"](area);
        );
    ''',
    'gym': '''
        (
          node["leisure"="fitness_centre"](area);
          way["leisure"="fitness_centre"](area);
          relation["leisure"="fitness_centre"](area);
          node["amenity"="gym"](area);
          way["amenity"="gym"](area);
          relation["amenity"="gym"](area);
        );
    ''',
    'library': '''
        (
          node["amenity"="library"](area);
          way["amenity"="library"](area);
          relation["amenity"="library"](area);
        );
    '''
}

# We already have the suburbs_data from the previous cell
# List to store all rows for the DataFrame
all_data = []

# Process each suburb for all amenity types
for i, suburb in enumerate(suburbs_data['elements']):
    suburb_id = suburb['id']
    suburb_name = suburb['tags'].get('name', f'Unnamed Suburb (ID: {suburb_id})')
    area_id = 3600000000 + suburb_id
    
    print(f"\n[{i+1}/{len(suburbs_data['elements'])}] Processing {suburb_name}...")
    
    # Query each amenity type for this suburb
    for category, amenity_query in amenity_queries.items():
        query = f"""
        [out:json][timeout:30];
        area({area_id});
        {amenity_query}
        out center;
        """
        
        try:
            response = requests.get(overpass_url, params={'data': query})
            response.raise_for_status()
            data = response.json()
            
            if data['elements']:
                for element in data['elements']:
                    name = element['tags'].get('name', f'Unnamed {category}')
                    
                    # Get coordinates
                    if 'center' in element:
                        lat, lon = element['center']['lat'], element['center']['lon']
                    else:
                        lat, lon = element['lat'], element['lon']
                    
                    # Add row to our data
                    all_data.append({
                        'suburb': suburb_name,
                        'category': category,
                        'name': name,
                        'lat': lat,
                        'lon': lon
                    })
                
                print(f"  + {category}: {len(data['elements'])}")
            else:
                print(f"  - {category}: 0")
                
        except requests.exceptions.RequestException as e:
            print(f"  ! Error fetching {category} for {suburb_name}: {e}")
        
        # Small delay to be polite to the API
        time.sleep(0.5)
    
    # Progress update every 20 suburbs
    if (i + 1) % 20 == 0:
        print(f"\n--- Progress: {i+1}/{len(suburbs_data['elements'])} suburbs processed ---")
        print(f"Total amenities collected so far: {len(all_data)}")

# Create DataFrame
print("\n=== Creating DataFrame ===")

if all_data:
    df = pd.DataFrame(all_data)
    
    print(f"\n=== FINAL RESULTS ===")
    print(f"Total records: {len(df)}")
    print(f"\nBreakdown by category:")
    print(df['category'].value_counts())
    
    print(f"\nBreakdown by suburb (top 10):")
    print(df['suburb'].value_counts().head(10))
    
    print(f"\nSample of the data:")
    print(df.head(10))
    
    # Save to CSV
    filename = f'melbourne_amenities_{datetime.now().strftime("%Y%m%d_%H%M%S")}.csv'
    df.to_csv(filename, index=False)
    print(f"\nData saved to: {filename}")
    
    # Make the DataFrame available for further analysis
    melbourne_amenities_df = df
    
    print(f"\nDataFrame is available as 'melbourne_amenities_df'")
    print(f"You can now analyze the data, e.g.:")
    print(f"- melbourne_amenities_df[melbourne_amenities_df['suburb'] == 'Brunswick']")
    print(f"- melbourne_amenities_df[melbourne_amenities_df['category'] == 'cafe'].head()")
    
else:
    print("No amenity data collected.")
    melbourne_amenities_df = pd.DataFrame()

print(f"\nCompleted at {datetime.now()}")

In [ ]:
overpass_url = "http://overpass-api.de/api/interpreter"

print("=== Getting all suburbs in Victoria, Australia ===")

# Query for all suburb-level administrative boundaries in Victoria
# We'll use a bounding box that covers the Greater Melbourne area
# Approximate bounding box for Greater Melbourne: 
# South: -38.5, West: 144.5, North: -37.3, East: 145.8
melbourne_bbox = "-38.2255,144.18,-37.5,145.375"

query_suburbs = f"""
[out:json][timeout:120];
(
  relation["admin_level"="10"]["boundary"="administrative"]({melbourne_bbox});
  relation["admin_level"="9"]["place"="suburb"]({melbourne_bbox});
  relation["place"="suburb"]({melbourne_bbox});
  // Ways tagged as suburbs
  way["place"="suburb"]({melbourne_bbox});    

  // Nodes tagged as suburbs  
  node["place"="suburb"]({melbourne_bbox});
);
out ids tags;
"""

print("Querying for all suburbs in the Melbourne metropolitan area...")
try:
    response = requests.get(overpass_url, params={'data': query_suburbs})
    response.raise_for_status()
    suburbs_data = response.json()
except requests.exceptions.RequestException as e:
    print(f"An error occurred while fetching suburbs: {e}")
    suburbs_data = {'elements': []}

print(len(suburbs_data['elements']))
print("\n=== List of Suburbs in Melbourne Metropolitan Area ===")
for element in suburbs_data['elements']:
    if 'tags' in element and 'name' in element['tags']:
        print(element['tags']['name'])

=== Getting all suburbs in Victoria, Australia ===
Querying for all suburbs in the Melbourne metropolitan area...
594

=== List of Suburbs in Melbourne Metropolitan Area ===
Heathmont
Dandenong
Lilydale
Wheelers Hill
Grovedale
Manifold Heights
Fyansford
Oakleigh South
Box Hill North
Carlton North
Strathmore
Blackburn South
Fitzroy North
Laverton
Doveton
Bulla
Surrey Hills
Oak Park
Reservoir
Drumcondra
McKinnon
Bentleigh
Kingsbury
Donnybrook
Briar Hill
Mont Albert North
Altona Meadows
Sandhurst
Mulgrave
Craigieburn
Ormond
Werribee South
Narre Warren South
Forest Hill
Avondale Heights
Maribyrnong
Panton Hill
Kangaroo Ground
Watsonia North
Bellfield
Heidelberg
Rosanna
Heidelberg West
Heidelberg Heights
Yan Yean
Mambourin
Greensborough
Essendon Fields
Northcote
Thornbury
Docklands
Malvern
St Kilda
Montmorency
Macleod
Watsonia
Fairfield
Alphington
Clifton Hill
Collingwood
Fitzroy
Abbotsford
East Melbourne
Richmond
Cremorne
Southbank
South Wharf
Port Melbourne
Tarneit
Hoppers Crossing
Point 

In [43]:
suburb_to_check = "Clayton"  # Replace with the suburb name you're looking for

# Function to check if suburb exists
def check_suburb_exists(suburbs_data, suburb_name):
    for element in suburbs_data['elements']:
        if 'tags' in element and 'name' in element['tags']:
            if element['tags']['name'].lower() == suburb_name.lower():
                return True
    return False

# Check if the suburb exists
if check_suburb_exists(suburbs_data, suburb_to_check):
    print(f"The suburb '{suburb_to_check}' exists in the data.")
else:
    print(f"The suburb '{suburb_to_check}' does not exist in the data.")

The suburb 'Clayton' exists in the data.


In [39]:
print("\n=== List of Suburbs in Melbourne Metropolitan Area ===")
for element in suburbs_data['elements']:
    if 'tags' in element and 'name' in element['tags']:
        print(element['tags']['name'])


=== List of Suburbs in Melbourne Metropolitan Area ===
Ivanhoe East
Ivanhoe
Alphington
Fairfield
Thornbury
Northcote
Clifton Hill
Abbotsford
Fitzroy North
Brunswick East
Collingwood
Fitzroy
Brunswick
Princes Hill
Carlton North
Carlton
Parkville
Melbourne
Balwyn North
Kew East
Deepdene
Kew
Canterbury
Camberwell
Hawthorn East
Balwyn
Hawthorn
Burnley
Richmond
East Melbourne
South Yarra
Cremorne
Toorak
Kooyong
Glen Iris
North Melbourne
Brunswick West
Templestowe
Surrey Hills
Burwood
Malvern
Ashburton
Ashwood
Armadale
Malvern East
Prahran
Caulfield North
Southbank
Windsor
Caulfield East
Travancore
St Kilda East
South Melbourne
Caulfield
South Wharf
Albert Park
St Kilda
Balaclava
Docklands
Kensington
Ripponlea
Middle Park
St Kilda West
Moonee Ponds
Flemington
Ascot Vale
Elsternwick
Ringwood North
West Melbourne
Elwood
Ringwood
Footscray
Port Melbourne
Maribyrnong
Seddon
Ringwood East
Yarraville
Spotswood
Maidstone
Croydon South
Newport
Kingsville
West Footscray
Williamstown
Croydon
South Ki